# CIFAR-10 image analysis using CNN
### Object Recognition with Convolutional Neural Networks in the Keras Deep Learning Library, by Jason Brownlee on July 1, 2016 in Deep Learning
##### code modified from: https://machinelearningmastery.com/object-recognition-convolutional-neural-networks-keras-deep-learning-library/
##### code modified from: TensorFlow+Keras[深度學習]人工智慧實務應用 / 林大貴

# 1. Import Library

In [29]:
# Simple CNN model for CIFAR-10

import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

ModuleNotFoundError: No module named 'tensorflow'

In [31]:
# for tensorflow 2.0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D, ZeroPadding2D
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import ModelCheckpoint

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
np.random.seed(3)

# 資料準備

In [ ]:
(x_train_image,y_train_label),(x_test_image,y_test_label) = tf.keras.datasets.cifar10.load_data()

In [ ]:
print("train data:",'images:',x_train_image.shape, " labels:",y_train_label.shape) 
print("test  data:",'images:',x_test_image.shape, " labels:",y_test_label.shape) 

## View the images

In [ ]:
fig = plt.gcf()
fig.set_size_inches(12,14)

for i in range(0,10):
    ax=plt.subplot(5,5,1+i)
    ax.imshow(x_train_image[i], cmap='binary')
    title= "label=" +str(y_train_label[i])
    ax.set_title(title,fontsize=10) 
    ax.set_xticks([]);ax.set_yticks([])        
plt.show()

## Prepare the data

In [ ]:
x_train_image_normalize = x_train_image.astype('float32') / 255.0
x_test_image_normalize = x_test_image.astype('float32') / 255.0

In [ ]:
y_train_label_OneHot = tf.keras.utils.to_categorical(y_train_label)
y_test_label_OneHot = tf.keras.utils.to_categorical(y_test_label)

In [ ]:
y_test_label_OneHot.shape

# 建立模型

In [ ]:
num_classes = y_test_label_OneHot.shape[1]

In [ ]:
# Create the model
model = Sequential()
model.add(Conv2D(32, (3, 3), input_shape=(32, 32, 3), padding='same', activation='relu'))
model.add(Dropout(0.2))
model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer='adam', metrics=['accuracy'])

# 載入之前訓練的模型

In [ ]:
# load keras file
from keras.models import load_model

try:
    model = load_model('SaveModel/my_model.keras')
    print("載入模型成功!繼續訓練模型")
except :    
    print("載入模型失敗!開始訓練一個新模型")

# 訓練模型

In [ ]:
print(model.summary())

In [ ]:
# saves the model weights after each epoch if the validation loss decreased
checkpointer = ModelCheckpoint(filepath='model_tmp.h5', monitor = "val_accuracy",
                               verbose=1, save_best_only=True)

In [ ]:
# tf.keras.callbacks.ModelCheckpoint(
#    filepath,
#    monitor: str = "val_loss",
#    verbose: int = 0,
#    save_best_only: bool = False,
#    save_weights_only: bool = False,
#    mode: str = "auto",
#    save_freq="epoch",
#    options=None,
#    initial_value_threshold=None,
#    **kwargs
#)

## 2023-09-01  
## (1) 在 ModelChckpoint 函式中有漏洞, 無法使用 x.keras 格式, 所以改用舊的 h5 格式.

In [ ]:
train_history=model.fit(x_train_image_normalize, y_train_label_OneHot,
                        validation_split=0.2,
                        epochs=40, batch_size=128, verbose=1, 
                        callbacks=[checkpointer])          

In [ ]:
import matplotlib.pyplot as plt
def show_train_history(train_acc,test_acc):
    plt.plot(train_history.history[train_acc])
    plt.plot(train_history.history[test_acc])
    plt.title('Train History')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['train', 'test'], loc='upper left')
    plt.show()

In [ ]:
show_train_history('accuracy','val_accuracy')

In [ ]:
show_train_history('loss','val_loss')

# 評估模型準確率

In [ ]:
scores = model.evaluate(x_test_image_normalize, 
                        y_test_label_OneHot, verbose=0)
scores[1]

# 進行預測

In [ ]:
predict_probability = model.predict(x_test_image_normalize)
prediction=np.argmax(predict_probability,axis=1)

In [ ]:
prediction[:10]

# 查看預測結果

In [ ]:
label_dict={0:"airplane",1:"automobile",2:"bird",3:"cat",4:"deer",
            5:"dog",6:"frog",7:"horse",8:"ship",9:"truck"}

In [ ]:
def plot_images_labels_prediction(images,labels,prediction,
                                  idx,num=10):
    fig = plt.gcf()
    fig.set_size_inches(12, 14)
    if num>25: num=25 
    for i in range(0, num):
        ax=plt.subplot(5,5, 1+i)
        ax.imshow(images[idx], cmap='binary')
        title= "label=" +str(labels[idx])
        if len(prediction)>0:
            title+=",predict="+str(prediction[idx]) 
            
        ax.set_title(title,fontsize=10) 
        ax.set_xticks([]);ax.set_yticks([])        
        idx+=1 
    plt.show()

In [ ]:
plot_images_labels_prediction(x_test_image,y_test_label, prediction,0,10)

# 查看預測機率

In [ ]:
Predicted_Probability=model.predict(x_test_image_normalize)

In [ ]:
def show_Predicted_Probability(y,prediction,
                               x_img,Predicted_Probability,i):
    print('label:',label_dict[y[i][0]],
          'predict:',label_dict[prediction[i]])
    plt.figure(figsize=(2,2))
    plt.imshow(np.reshape(x_test_image[i],(32, 32,3)))
    plt.show()
    for j in range(10):
        print(label_dict[j]+
              ' Probability:%1.9f'%(Predicted_Probability[i][j]))

In [ ]:
show_Predicted_Probability(y_test_label,prediction,
                           x_test_image,Predicted_Probability,1)

In [ ]:
show_Predicted_Probability(y_test_label,prediction,
                           x_test_image,Predicted_Probability,6)

# confusion matrix

In [ ]:
import pandas as pd
print(label_dict)
pd.crosstab(y_test_label.reshape(-1),prediction,
            rownames=['label'],colnames=['predict'])

In [ ]:
print(label_dict)

# Save and load model 

In [ ]:
# save model to keras file
model.save('my_model.keras')  

In [ ]:
del model  # deletes the existing model

In [ ]:
from keras.models import load_model
model = load_model('my_model.keras')
model.summary()

# Load model from Check Point Call Back

In [ ]:
del model  # deletes the existing model

## 2023-09-01
## (2) 在 <font color='red'>load_model()</font> 使用 h5 格式時, 這個檔案上層的目錄夾, 不能有中文.
## (3) 如果看到有 <font color='red'>UTF-8</font> error message, 請檢查這個檔案上層的目錄夾, 將中文改成英文.

In [ ]:
from keras.models import load_model
model = load_model('model_tmp.h5')
model.summary()

In [ ]:
Predicted_Probability=model.predict(x_test_image_normalize)

In [ ]:
show_Predicted_Probability(y_test_label,prediction,
                           x_test_image,Predicted_Probability,1)